In [6]:
import os
import torch
import numpy as np
from wm_gym_env import (
    MatrixGenerationArgs, VLMRewardArgs, seed_everything,
    OpenAIRewardModel, load_matrix_gym_pipe, matrixGym, export_to_video
)

import imageio
import tempfile
from PIL import Image
from IPython.display import Video, display

def create_temp_video(pil_images, fps=4):
    with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as temp:
        writer = imageio.get_writer(temp.name, fps=fps)

        for img in pil_images:
            frame = np.array(img)  # Convert PIL.Image to numpy array
            writer.append_data(frame)

        writer.close()
        return temp.name

In [2]:
matrix_gen_config = MatrixGenerationArgs(
    prompt="On a lush green meadow, a white car is driving. From an overhead panoramic shot, \
            this car is adorned with blue and red stripes on its body, and it has a black spoiler at the rear. \
            The camera follows the car as it moves through a field of golden wheat, surrounded by green grass and trees. \
            In the distance, a river and some hills can be seen, with a cloudless blue sky above.",
    model_path="/mnt/d/model_ckpts_stage4/model_ckpts_stage4/stage3",
    video_path="/home/andy/matrix/base_video.mp4",
)

vlm_rm_config = VLMRewardArgs(
    model="gpt-4o",
    api_key="debug",  # Replace with your actual API key if testing live
    reward_query=(
        "You are a video analyst. Your task is to analyze a sequence of consecutive images and describe the spatial relationship "
        "between the car and any potential obstacles. Based on this analysis, assess the risk of a possible collision."
    ),
    reward_criteria=(
        "Here is the video description: {} "
        "Return 1 if there is no risk of collision with any obstacle. "
        "Return 0 if there is a potential risk of collision, but no collision has occurred yet. "
        "Return -1 if you believe a collision has already occurred between the car and an obstacle, regardless of whether there was damage."
        "Your response must only contain one of the following: 1, 0, or -1. Do not include any additional explanation or description."
    ),
)


In [3]:
seed_everything(matrix_gen_config.seed)
debug_clip_output_dir = "./debug_clip_output"
os.makedirs(debug_clip_output_dir, exist_ok=True)


In [4]:
gpt4_rm = OpenAIRewardModel(**vars(vlm_rm_config))
wm_gym_pipe = load_matrix_gym_pipe(matrix_gen_config)
env = matrixGym(wm_gym_pipe, gpt4_rm)


The config attributes {'invert_scale_latents': False} were passed to AutoencoderKLCogVideoX, but are not expected and will be ignored. Please verify your config.json configuration file.
Loading pipeline components...: 100%|██████████| 5/5 [00:55<00:00, 11.07s/it]


Latent size: torch.Size([1, 5, 16, 60, 90])


In [9]:
state, info = env.reset()
video_clip = env.render(mode="pil")
export_to_video(video_clip, fps=matrix_gen_config.fps,
                output_video_path=os.path.join(debug_clip_output_dir, "0_reset_video.mp4"))
video_path = create_temp_video(video_clip)
display(Video(video_path, embed=True))


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
state, reward, terminated, truncated, info = env.step("DL")
video_clip = env.render(mode="pil")
print("Step 1 — DL:")
print("Reward:", reward)
print("VLM Response:", info)
export_to_video(video_clip, fps=matrix_gen_config.fps,
                output_video_path=os.path.join(debug_clip_output_dir, "1_DL_video.mp4"))
video_path = create_temp_video(video_clip)
display(Video(video_path, embed=True))

